# 03 — Modèles : 2 VLM × 3 précisions
On lance les 6 configurations sur **les mêmes images**. La mémoire est libérée entre chaque modèle.

Les résultats vont dans Google Drive. **Si Colab coupe la session**, relancez toutes les cellules : les configurations déjà terminées sont sautées.

In [ ]:
# Installation (Google Colab / Kaggle avec GPU)
!git clone https://github.com/doaemalak/VLMs-l-gers-pour-l-gender-ou-juger-des-images.git
%cd VLMs-l-gers-pour-l-gender-ou-juger-des-images
!pip install -q -r requirements.txt

# Colab efface tout entre deux sessions : on retélécharge les 500 mêmes images (seed 42)
![ -f data/subset/subset.csv ] || python data/download.py --config configs/data.yaml

In [ ]:
# Les résultats sont enregistrés dans Google Drive (Mon Drive/projet7_results)
# pour ne pas être perdus si Colab coupe la session.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/projet7_results
!rm -rf results && ln -s /content/drive/MyDrive/projet7_results results
!ls results

In [ ]:
import os
from src.run_captions import run

LIMIT = None   # None = les 500 images ; mettre 20 pour un test rapide

for cfg in ['configs/vlm1.yaml', 'configs/vlm2.yaml']:
    name = os.path.basename(cfg).replace('.yaml', '')
    for precision in ['normal', '8bit', '4bit']:
        if os.path.exists(f'results/captions/{name}_{precision}.json'):
            print(f'{name} {precision} : déjà fait, on passe')
            continue
        print(f'=== {name} {precision} ===')
        run(cfg, precision, limit=LIMIT)

## Tableau comparatif (construit à partir de tous les résultats enregistrés)

In [ ]:
import glob, json
import pandas as pd
rows = [json.load(open(f)) for f in sorted(glob.glob('results/captions/*.json'))]
table = pd.DataFrame(rows)
table.to_csv('results/tableau_comparatif.csv', index=False)
table